# Limpieza de población por edad quinquenal (INE)

En este notebook se realiza la limpieza del dataset de población por grupos de edad quinquenales
para cada municipio y año.

## 1. Carga de librerías

In [1]:
import pandas as pd
import numpy as np

## 2. Carga del dataset

In [2]:
df_edad_raw = pd.read_csv("../data_raw/edades.csv", sep=";", low_memory=False)

## 3. Exploración inicial

In [3]:
df_edad_raw.head()

,Sexo,Municipios,Edad (grupos quinquenales),Periodo,Total
0,Total,Total Nacional,Todas las edades,1 de enero de 2022,47.475.420
1,Total,Total Nacional,Todas las edades,1 de enero de 2021,47.385.107
2,Total,Total Nacional,Todas las edades,1 de enero de 2020,47.450.795
3,Total,Total Nacional,Todas las edades,1 de enero de 2019,47.026.208
4,Total,Total Nacional,Todas las edades,1 de enero de 2018,46.722.980


In [4]:
df_edad_raw.shape

(10739520, 5)

In [5]:
df_edad_raw.dtypes

Sexo                          object
Municipios                    object
Edad (grupos quinquenales)    object
Periodo                       object
Total                         object
dtype: object

In [6]:
df_edad_raw['Total'].isna().sum()

np.int64(21978)

## 4. Normalizar nombres de columnas

In [7]:
df_edad_raw.columns = (
    df_edad_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df_edad_raw.columns

Index(['sexo', 'municipios', 'edad_(grupos_quinquenales)', 'periodo', 'total'], dtype='object')

In [8]:
df_edad_raw = df_edad_raw.rename(columns={"edad_(grupos_quinquenales)": "edad_grupo"})

## 5. Exploración de variables clave

In [9]:
# Valores de sexo
df_edad_raw['sexo'].value_counts()

sexo
Total      3579840
Hombres    3579840
Mujeres    3579840
Name: count, dtype: int64

In [10]:
# Valores de edad
df_edad_raw["edad_grupo"].unique()[:20]

array(['Todas las edades', 'De 0 a 4 años', 'De 5 a 9 años',
       'De 10 a 14 años', 'De 15 a 19 años', 'De 20 a 24 años',
       'De 25 a 29 años', 'De 30 a 34 años', 'De 35 a 39 años',
       'De 40 a 44 años', 'De 45 a 49 años', 'De 50 a 54 años',
       'De 55 a 59 años', 'De 60 a 64 años', 'De 65 a 69 años',
       'De 70 a 74 años', 'De 75 a 79 años', 'De 80 a 84 años',
       'De 85 a 89 años', 'De 90 a 94 años'], dtype=object)

In [11]:
# Valores de periodo
df_edad_raw['periodo'].unique()[:20]

array(['1 de enero de 2022', '1 de enero de 2021', '1 de enero de 2020',
       '1 de enero de 2019', '1 de enero de 2018', '1 de enero de 2017',
       '1 de enero de 2016', '1 de enero de 2015', '1 de enero de 2014',
       '1 de enero de 2013', '1 de enero de 2012', '1 de enero de 2011',
       '1 de enero de 2010', '1 de enero de 2009', '1 de enero de 2008',
       '1 de enero de 2007', '1 de enero de 2006', '1 de enero de 2005',
       '1 de enero de 2004', '1 de enero de 2003'], dtype=object)

## 6. Filtrado de población total

In [12]:
# Como en el padrón, no necesitamos separar por sexo, ya que solo queremos la población total por edad
df_edad = df_edad_raw[df_edad_raw['sexo'] == 'Total'].copy()

In [13]:
df_edad.shape

(3579840, 5)

## 7. Eliminación de "Todas las edades"

In [14]:
df_edad = df_edad[df_edad['edad_grupo'] != "Todas las edades"]

In [15]:
df_edad['edad_grupo'].unique()[:21]

array(['De 0 a 4 años', 'De 5 a 9 años', 'De 10 a 14 años',
       'De 15 a 19 años', 'De 20 a 24 años', 'De 25 a 29 años',
       'De 30 a 34 años', 'De 35 a 39 años', 'De 40 a 44 años',
       'De 45 a 49 años', 'De 50 a 54 años', 'De 55 a 59 años',
       'De 60 a 64 años', 'De 65 a 69 años', 'De 70 a 74 años',
       'De 75 a 79 años', 'De 80 a 84 años', 'De 85 a 89 años',
       'De 90 a 94 años', 'De 95 a 99 años', '100 y más años'],
      dtype=object)

## 8. Limpieza de la columna 'periodo'

In [16]:
# Para el analálisis solo necesitamos el año
df_edad['periodo'] = df_edad['periodo'].str[-4:].astype(int)

In [17]:
df_edad['periodo'].unique()[:20]

array([2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012,
       2011, 2010, 2009, 2008, 2007, 2006, 2005, 2004, 2003])

## 9. Limpiamos `poblacion`

In [18]:
df_edad['total'] = (
    df_edad['total']
    .astype(str)
    .str.replace(".", "", regex=False)
)

In [19]:
df_edad['total'] = pd.to_numeric(df_edad['total'], errors="coerce")

## 10. Tratamiento de valores faltantes

### Valores faltantes en población

Al revisar la columna `total` se detectaron algunos valores nulos (NaN).
Estos casos corresponden a municipios y años en los que no hay dato de población
disponible en el dataset original.

Dado que no es posible analizar la evolución de la población sin ese valor,
se decidió eliminar estas observaciones del conjunto de datos.

In [20]:
# Comprobamos si hay valores nulos para ver si hay valores que no se pudieron convertir a número
df_edad['total'].isna().sum()

np.int64(6993)

In [21]:
# Eliminamos las filas sin datos de población
df_edad = df_edad.dropna(subset=['total'])

In [22]:
# Verificamos que ya no quedan nulos
df_edad['total'].isna().sum()

np.int64(0)

## 11. Separación de código y nombre del municipio

In [23]:
# Eliminamos agregados territoriales como 'Total Nacional'
df_edad = df_edad[df_edad['municipios'] != "Total Nacional"]

In [24]:
# Separamos el código INE del nombre del municipio
df_edad[['codigo_municipio', 'municipio']] = df_edad['municipios'].str.split(" ", n=1, expand=True)

In [25]:
# Reordenamos columnas
cols = ["codigo_municipio", "municipio"] + [col for col in df_edad.columns if col not in ["codigo_municipio", "municipio"]]
df_edad = df_edad[cols]

In [26]:
df_edad.head()

,codigo_municipio,municipio,sexo,municipios,edad_grupo,periodo,total
460,44001,Ababuj,Total,44001 Ababuj,De 0 a 4 años,2022,0.0
461,44001,Ababuj,Total,44001 Ababuj,De 0 a 4 años,2021,0.0
462,44001,Ababuj,Total,44001 Ababuj,De 0 a 4 años,2020,0.0
463,44001,Ababuj,Total,44001 Ababuj,De 0 a 4 años,2019,1.0
464,44001,Ababuj,Total,44001 Ababuj,De 0 a 4 años,2018,2.0


## 12. Transformación de grupos de edad

In [27]:
# Extraemos la edad inicial del grupo
df_edad['edad_inicio'] = df_edad['edad_grupo'].str.extract(r"(\d+)")[0]

In [28]:
# Convertimos a númerico
df_edad['edad_inicio'] = pd.to_numeric(df_edad['edad_inicio'], errors="coerce")

In [29]:
df_edad[['edad_grupo', 'edad_inicio']].head()

,edad_grupo,edad_inicio
460,De 0 a 4 años,0
461,De 0 a 4 años,0
462,De 0 a 4 años,0
463,De 0 a 4 años,0
464,De 0 a 4 años,0


In [30]:
df_edad["edad_inicio"].unique()[:21]

array([  0,   5,  10,  15,  20,  25,  30,  35,  40,  45,  50,  55,  60,
        65,  70,  75,  80,  85,  90,  95, 100])

## 13. Limpieza final de columnas

In [31]:
# Eliminamos columnas que ya no aportan información
df_edad = df_edad.drop(columns=['municipios', 'sexo'])

In [32]:
df_edad.columns, df_edad.head(100)

(Index(['codigo_municipio', 'municipio', 'edad_grupo', 'periodo', 'total',
        'edad_inicio'],
       dtype='object'),
     codigo_municipio municipio       edad_grupo  periodo  total  edad_inicio
 460            44001    Ababuj    De 0 a 4 años     2022    0.0            0
 461            44001    Ababuj    De 0 a 4 años     2021    0.0            0
 462            44001    Ababuj    De 0 a 4 años     2020    0.0            0
 463            44001    Ababuj    De 0 a 4 años     2019    1.0            0
 464            44001    Ababuj    De 0 a 4 años     2018    2.0            0
 ..               ...       ...              ...      ...    ...          ...
 555            44001    Ababuj  De 20 a 24 años     2007    3.0           20
 556            44001    Ababuj  De 20 a 24 años     2006    3.0           20
 557            44001    Ababuj  De 20 a 24 años     2005    2.0           20
 558            44001    Ababuj  De 20 a 24 años     2004    3.0           20
 559            440

In [33]:
# Renombramos columna de población 
df_edad = df_edad.rename(columns={'total': 'poblacion'})

In [34]:
df_edad.columns

Index(['codigo_municipio', 'municipio', 'edad_grupo', 'periodo', 'poblacion',
       'edad_inicio'],
      dtype='object')

In [35]:
df_edad['codigo_municipio'] = df_edad['codigo_municipio'].astype(str).str.zfill(5)

## 14. Comprobación de duplicados

In [36]:
# Comprobamos si hay duplicados en municipio, año y grupo de edad
df_edad.duplicated(subset=['codigo_municipio', 'periodo', 'edad_inicio']).sum()

np.int64(0)

## 15. Ordenar el dataset

In [37]:
# Ordenamos el dataset
df_edad = df_edad.sort_values(['codigo_municipio', 'periodo', 'edad_inicio'])

In [38]:
df_edad.head()

,codigo_municipio,municipio,edad_grupo,periodo,poblacion,edad_inicio
156679,01001,Alegría-Dulantzi,De 0 a 4 años,2003,125.0,0
156699,01001,Alegría-Dulantzi,De 5 a 9 años,2003,74.0,5
156719,01001,Alegría-Dulantzi,De 10 a 14 años,2003,81.0,10
156739,01001,Alegría-Dulantzi,De 15 a 19 años,2003,71.0,15
156759,01001,Alegría-Dulantzi,De 20 a 24 años,2003,98.0,20


In [39]:
# Reordenamos columnas en un orden lógico
df_edad = df_edad[['codigo_municipio', 'municipio', 'periodo', 'edad_grupo', 'edad_inicio', 'poblacion']]

In [40]:
df_edad.head()

,codigo_municipio,municipio,periodo,edad_grupo,edad_inicio,poblacion
156679,01001,Alegría-Dulantzi,2003,De 0 a 4 años,0,125.0
156699,01001,Alegría-Dulantzi,2003,De 5 a 9 años,5,74.0
156719,01001,Alegría-Dulantzi,2003,De 10 a 14 años,10,81.0
156739,01001,Alegría-Dulantzi,2003,De 15 a 19 años,15,71.0
156759,01001,Alegría-Dulantzi,2003,De 20 a 24 años,20,98.0


## 17. Exportación del dataset limpio

In [41]:
df_edad.to_csv("../data_processed/edad_limpio.csv", index=False)